In [ ]:
import json
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
path = r'/path/to/datasets/DATASET_UUID_REDACTED/eforms.json' #train
path1= r'/path/to/datasets/DATASET_UUID_REDACTED/eforms.json'  #val
path2= r'/path/to/datasets/DATASET_UUID_REDACTED/eforms.json'  #test

In [ ]:
def flatten_dict(d, parent_key='', sep='_'):
    """Flatten nested dictionaries."""
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_dict(v, new_key, sep=sep).items())
        else:
            items.append((new_key, v))
    return dict(items)

In [ ]:
def preprocess_json(json_path):
    with open(json_path, 'r') as f:
        eforms = json.load(f)

    processed_data = []
    for eform in eforms:
        # Retain only the first 3 pages
        eform['eForm']['pages'] = eform['eForm']['pages'][:3]

        # Remove specific fields from page 0
        inclusion_criteria_data = eform['eForm']['pages'][0]['page_data']
        del inclusion_criteria_data['difference_diagnosis_treatment']
        
        # Reserve specific fields from page 1
        patient_data = eform['eForm']['pages'][0]['page_data']
        
        # Remove specific fields from page 2
        diagnosis_data = eform['eForm']['pages'][2]['page_data']
        del diagnosis_data['distance_anorectal_mr']
        del diagnosis_data['mesorectal_invation_mr']
        del diagnosis_data['ex_anal_sphincter_invation_mr']
        del diagnosis_data['int_anal_sphincter_invation_mr']
        del diagnosis_data['extramural_invation_mr']
        del diagnosis_data['tumor_clinical_category']
        del diagnosis_data['regional_nodes_clinical_category']
        del diagnosis_data['metastasis_clinical_category']
        del diagnosis_data['clinical_stage_group']
        del diagnosis_data['clinical_stage_group_user']
 
        del diagnosis_data['date_pathology_sample']
        del diagnosis_data['tumor_histotype']
        del diagnosis_data['circ_resection_margin']
        del diagnosis_data['perineural_invasion']
        del diagnosis_data['vascular_invasion']
        del diagnosis_data['regression_grade']
        del diagnosis_data['microsatelite_instability']
 
        eform['eForm']['pages'][2]['lists'] = []

        # # Combine the remaining data into a single dictionary
        # combined_data = {
        #     inclusion_criteria_data,
        #     diagnosis_data
        # }
        
        # combined_data['EVI'] = diagnosis_data['extramural_invation_mr']['value']
        # combined_data['MFI'] = diagnosis_data['mesorectal_invation_mr']['value']
        # processed_data.append(combined_data)

    return inclusion_criteria_data, patient_data, diagnosis_data

# Preprocess the JSON data
inclusion_criteria_data, patient_data, diagnosis_data = preprocess_json(path)
print(diagnosis_data)
# print(data.head())

In [ ]:
print(inclusion_criteria_data)

In [ ]:
print(inclusion_criteria_data['age_at_diagnosis'])

In [ ]:
print(patient_data)

In [ ]:
# 1. Read and preprocess the JSON file
def preprocess_json(json_path):
    with open(json_path, 'r') as f:
        eforms = json.load(f)

    processed_data = []
    for eform in eforms:
        # Retain only the first 3 pages
        eform['eForm']['pages'] = eform['eForm']['pages'][:3]

        # Remove specific fields from page 1
        inclusion_criteria_data = eform['eForm']['pages'][0]['page_data']
        del inclusion_criteria_data['difference_diagnosis_treatment']
 
        diagnosis_data = eform['eForm']['pages'][2]['page_data']
        del diagnosis_data['distance_anorectal_mr']
        del diagnosis_data['mesorectal_invation_mr']
        del diagnosis_data['ex_anal_sphincter_invation_mr']
        del diagnosis_data['int_anal_sphincter_invation_mr']
        del diagnosis_data['extramural_invation_mr']
        del diagnosis_data['tumor_clinical_category']
        del diagnosis_data['regional_nodes_clinical_category']
        del diagnosis_data['metastasis_clinical_category']
        del diagnosis_data['clinical_stage_group']
        del diagnosis_data['clinical_stage_group_user']
 
        del diagnosis_data['date_pathology_sample']
        del diagnosis_data['tumor_histotype']
        del diagnosis_data['circ_resection_margin']
        del diagnosis_data['perineural_invasion']
        del diagnosis_data['vascular_invasion']
        del diagnosis_data['regression_grade']
        del diagnosis_data['microsatelite_instability']
 
        eform['eForm']['pages'][2]['lists'] = []

        # Combine the remaining data into a single dictionary
        combined_data = {
            inclusion_criteria_data,
            diagnosis_data
        }
        
        combined_data['EVI'] = diagnosis_data['extramural_invation_mr']['value']
        combined_data['MFI'] = diagnosis_data['mesorectal_invation_mr']['value']
        processed_data.append(combined_data)

    return pd.DataFrame(processed_data)

# Preprocess the JSON data
data = preprocess_json(path)
print("Preprocessed Data:")
print(data.head())

In [ ]:
# Preprocess features for the model
def preprocess_features(data, target_columns):
    # Separate features and targets
    X = data.drop(columns=target_columns)
    y = data[target_columns]

    # Convert boolean features to numeric
    X = X.astype({col: 'int' for col in X.select_dtypes(include=['bool']).columns})

    # Encode categorical string features as numbers
    from sklearn.preprocessing import LabelEncoder
    for col in X.select_dtypes(include=['object']).columns:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col])

    # Ensure only numeric and boolean features remain
    X = X.select_dtypes(include=['number', 'bool'])
    
    return X, y


In [ ]:
# Preprocess features and targets
X, y = preprocess_features(data, target_columns=['EVI', 'MFI'])

# Train-test split
X_train, X_test, y_train_evi, y_test_evi = train_test_split(X, y['EVI'], test_size=0.2, random_state=42)
X_train, X_test, y_train_mfi, y_test_mfi = train_test_split(X, y['MFI'], test_size=0.2, random_state=42)

# Train EVI model
evi_model = RandomForestClassifier(random_state=42)
evi_model.fit(X_train, y_train_evi)
y_pred_evi = evi_model.predict(X_test)

# Evaluate EVI model
print("EVI Classification Report:")
print(classification_report(y_test_evi, y_pred_evi))

# Train MFI model
mfi_model = RandomForestClassifier(random_state=42)
mfi_model.fit(X_train, y_train_mfi)
y_pred_mfi = mfi_model.predict(X_test)

# Evaluate MFI model
print("MFI Classification Report:")
print(classification_report(y_test_mfi, y_pred_mfi))


In [ ]:
#ask chatgpt how to select clinial feature about 231 cases train dataset